# Visualise complex functions
Draws the complex functions that make up the hybridisation functions R_jk(ω) (mechanical and optical susceptibilities and their combinations) as curves in the complex plane over a frequency sweep.

Runtime: about 10-60 s.

# START

# Define complex plotting function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import namedtuple
from levisim import *

set_plt_rcParams_to_xmgrace_style()

ComplexFunc = namedtuple(
    'ComplexFunc',
    [
        'func',
        'ω_range',
        'latex',
        'units_str',
        'resonance',
        'resonance_latex',
        'damping',
        'damping_latex',
    ])

def visualise_complex_func(complex_func, **kwargs):
    """
    Plot an instance of the namedtuple `ComplexFunc`
    """
    cf = complex_func
    ω_range = cf.ω_range

    msg = f'🍭 Plotting complex function "{cf.func}" over spectral reange {ω_range.min() / kHz=:.3f} to {ω_range.max() / kHz=:.3f}'
    print(msg), logger.info(msg)

    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3 * len(mosaic)), dpi=200)
    ax_dict = fig.subplot_mosaic(mosaic)

    vals = np.array([cf.func(ω) for ω in ω_range])
    xlims = ω_range.min() / kHz, ω_range.max() / kHz
    ylim = 1.2 * np.abs(vals).max()
    ylims = -ylim, +ylim
    ax = ax_dict['1']
    ax.set_title(rf'${cf.latex}$')
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(cf.units_str)
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.axhline(0, color=black, alpha=0.1)
    ax.axvline(0, color=black, alpha=0.1)
    ax.plot(ω_range / kHz, np.real(vals), label=rf'Re[${cf.latex}$]', alpha=0.4, lw=2.5)
    ax.plot(ω_range / kHz, np.imag(vals), label=rf'Im[${cf.latex}$]', alpha=0.4, lw=1.5, ls='--')
    ax.scatter(ω_range / kHz, np.abs(vals), label=rf'Abs[${cf.latex}$]', s=10, marker='o', alpha=0.3, c=ω_range, cmap='gist_rainbow')

    if cf.resonance and cf.damping:
        resonance = cf.resonance / kHz
        damping = cf.damping / kHz
        ax.axvline(resonance, color=RGB_green, lw=3, alpha=0.25, label=rf'${cf.resonance_latex}$')
        ax.axvline(resonance + damping / 2, color=purple, lw=3, alpha=0.25, label=rf'${cf.resonance_latex}\pm{cf.damping_latex}$')
        ax.axvline(resonance - damping / 2, color=purple, lw=3, alpha=0.25)

    ax.grid(alpha=0.1)
    ax.legend(loc='lower left', prop=dict(size=6))

    xlims, ylims = get_latex_doc_lims(0.3, ylim)
    ax = ax_dict['2']
    ax.set_title(rf'${cf.latex}$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.set_xlabel('Real part ' + cf.units_str)
    ax.set_ylabel('Imag. part ' + cf.units_str)
    ax.axhline(0, color=black, alpha=0.1)
    ax.axvline(0, color=black, alpha=0.1)
    ax.scatter(np.real(vals), np.imag(vals), s=10, marker='o', alpha=0.3, c=ω_range, cmap='gist_rainbow')
    ax.grid(alpha=0.1)

    plt.show()


# Optical susceptibilites

---

For photon annihilation $\chi_\mathrm{c}(\omega)$ and creation $\chi_\mathrm{c}^*(-\omega)$ inside the cavity

- Units: [$\chi_\mathrm{c}(\omega)$] = ($2\pi \cdot$ Hz)$^{-1}$
- Hermitian conjugate: $[\chi_\mathrm{c}(\omega)]^\dagger = \chi_\mathrm{c}^*(-\omega)$

$$ \chi_\mathrm{c}(\omega) \equiv \left[ -i(\omega + \Delta) + \tfrac{\kappa}{2} \right]^{-1} $$
$$ \chi_\mathrm{c}^*(-\omega) = \left[ -i(\omega - \Delta) + \tfrac{\kappa}{2} \right]^{-1} $$

---

Hermitian product of optical susceptibilities $\xi_\mathrm{c}$
- Units: [$\xi_\mathrm{c}$] = ($2\pi \cdot$ Hz)$^{-2}$
- Hermitian: $\xi_\mathrm{c}^\dagger = \xi_\mathrm{c}$

$$
    \xi_\mathrm{c}
    \equiv
    \chi_\mathrm{c}(\omega) \chi_\mathrm{c}^*(-\omega)
    =
    \frac{1}{\Delta^2 - \omega^2 +\frac{\kappa^2}{4} -i \kappa \omega}
$$

---

Hermitian difference of optical susceptibilities
- Units: [$\eta_\mathrm{c}$] = ($2\pi \cdot$ Hz)$^{-1}$
- Anti-Hermitian: $\eta_\mathrm{c}^\dagger = -\eta_\mathrm{c}$
- $\eta_\mathrm{c} = 2i\Delta \xi_\mathrm{c}$
$$
    \eta_\mathrm{c}
    \equiv
    \chi_\mathrm{c}(\omega) - \chi_\mathrm{c}^*(-\omega) 
    =
    \frac{2i\Delta}{\Delta^2 - \omega^2 +\frac{\kappa^2}{4} -i \kappa \omega}
$$

---

In [ ]:
N, d = 2, 2
ω_range = np.arange(-1000, 1000 + 1, 1) * kHz

mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=ω_range,
    gas_kwargs=DEFAULT_gas_kwargs,
    cavity_kwargs=DEFAULT_cavity_kwargs,
    toy_params=OFF_NODE_toy_params_pairs,
)

χC = ComplexFunc(
    mtrap.cavity.χC, ω_range,
    '\chi_\mathregular{c}(\omega)', latex_ω_units[-1],
    -mtrap.cavity.Δ, '-\Delta',
    mtrap.cavity.κ, '\kappa',
)
χC_star = ComplexFunc(
    mtrap.cavity.χC_star, ω_range,
    '\chi^*_\mathregular{c}(-\omega)', latex_ω_units[-1],
    mtrap.cavity.Δ, '+\Delta',
    mtrap.cavity.κ, '\kappa',
)

ηm = ComplexFunc(
    mtrap.cavity.ηm, ω_range,
    '\eta^-_\mathregular{c}', latex_ω_units[-1],
    None, None, None, None,
)
ηp = ComplexFunc(
    mtrap.cavity.ηp, ω_range,
    '\eta^+_\mathregular{c}', latex_ω_units[-1],
    None, None, None, None,
)

for complex_func in [
    χC, χC_star,
    ηm, ηp,
]:
    visualise_complex_func(complex_func)

# Mechanical susceptibilities

---

For phonon annihilation $\chi_j(\omega)$ and creation $\chi_j^*(-\omega)$ of $j^\mathrm{th}$ mode
- Units: [$\chi_j(\omega)$] = ($2\pi \cdot$ Hz)$^{-1}$
- Hermitian conjugate: $[\chi_j(\omega)]^\dagger = \chi_j^*(-\omega)$
$$ \chi_j(\omega) \equiv \left[ -i(\omega - \Omega_j) + \tfrac{\Gamma_j}{2} \right]^{-1} $$
$$ \chi_j^*(-\omega) = \left[ -i(\omega + \Omega_j) + \tfrac{\Gamma_j}{2} \right]^{-1} $$

---

Hermitian difference of mechanical susceptibilities
- Units: [$\mu_j$] = ($2\pi \cdot$ Hz)$^{-1}$
- Anti-Hermitian: $\mu_j^\dagger = -\mu_j$
$$
    \mu_j
    \equiv
    \chi_j(\omega) - \chi_j^*(-\omega) = 2i\Omega_j \chi_j(\omega) \chi_j^*(-\omega)
    =
    \frac{-2i\Omega_j}{\Omega_j^2 - \omega^2 +\frac{\Gamma_j^2}{4} -i \Gamma_j \omega}
$$

---


In [ ]:
N, d = 2, 2
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    # ω_range=ω_range,
    gas_kwargs=DEFAULT_gas_kwargs,
    cavity_kwargs=DEFAULT_cavity_kwargs,
    toy_params=OFF_NODE_toy_params_pairs,
)

for dim in list(mtrap.dimensions.values()):
    step = 5e-9 * kHz
    width = 3 * dim.Γ

    ω_range = np.arange(dim.Ω - width, dim.Ω + width + step, step)
    χj = ComplexFunc(
        dim.χ, ω_range,
        dim.latex_χ, latex_ω_units[-1],
        dim.Ω, f'+{dim.latex_Ω}',
        dim.Γ, dim.latex_Γ,
    )

    ω_range = np.arange(-dim.Ω - width, -dim.Ω + width + step, step)
    χj_star = ComplexFunc(
        dim.χ_star, ω_range,
        dim.latex_χ_star, latex_ω_units[-1],
        -dim.Ω, f'-{dim.latex_Ω}',
        dim.Γ, dim.latex_Γ,
    )

    ω_range = np.concatenate((
        np.arange(-450, -320, 1) * kHz,
        np.arange(-320, -280, 0.01) * kHz,
        np.arange(-280, 280, 1) * kHz,
        np.arange(280, 320, 0.01) * kHz,
        np.arange(320, 450 + 0.01, 0.01) * kHz,
    ))

    μm = ComplexFunc(
        dim.μm, ω_range,
        dim.latex_μm, latex_ω_units[-1],
        None, None, None, None,
    )
    μp = ComplexFunc(
        dim.μp, ω_range,
        dim.latex_μp, latex_ω_units[-1],
        None, None, None, None,
    )

    for complex_func in [
        χj, χj_star,
        μm, μp,
    ]:
        visualise_complex_func(complex_func)


# Hybridisation terms

In [ ]:
from functools import partial

N, d = 2, 2
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    # ω_range=ω_range,
    gas_kwargs=DEFAULT_gas_kwargs,
    cavity_kwargs=DEFAULT_cavity_kwargs,
    toy_params=OFF_NODE_toy_params_pairs,
)

for dim_j in list(mtrap.dimensions.values()):
    for dim_k in list(mtrap.dimensions.values()):
        if dim_j == dim_k:
            continue
        Rjk = HybridisationTerm(sign=1, pairs=[(dim_j.label, dim_k.label)])
        # latex_Rjk = rf'R_{{{dim_j.latex + dim_k.latex}}}'

        # avoid the + sign
        latex_Rjk = term_latex(Rjk)[1:]

        min_resonance = min(dim_j.Ω, dim_k.Ω)
        max_resonance = max(dim_j.Ω, dim_k.Ω)


        step = 0.5*kHz
        width = 100*kHz
        ω_range = np.arange(min_resonance - width, max_resonance + width + step, step)

        Rjk_complex_func = ComplexFunc(
            partial(mtrap.hybridisation_term_value, Rjk), ω_range,
            latex_Rjk, '',
            None, None, None, None,
        )
        visualise_complex_func(Rjk_complex_func)



# END